# Amazon-Reviews-2023 → ABSA smartphone corpus

**Phase 1a (this notebook / Colab):** load Hugging Face, keep real smartphones, take their reviews, run methodology Step 1. Download `hf_corpus.zip`.

**Phase 1b–2 (your PC):** import the zip into SQLite, run Steps 2–6, open the website.

```powershell
python run.py import-corpus .\hf_corpus --replace --analyze
python run.py serve
```

Open http://127.0.0.1:8000/ui/ — **Recommend** and **Phones** only.

If your PC internet is fine, skip Colab:

```text
python run.py prepare-corpus --max-phones 40 --max-reviews 60 --max-review-scan 0
python run.py serve
```

`--max-review-scan 0` scans the review JSONL until selected phones have reviews or the file ends.

Dataset: [`McAuley-Lab/Amazon-Reviews-2023`](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) (Cell Phones & Accessories). Accessories are filtered out. Review `body` is exact dataset text.

### Outputs (zipped for download)
- `phones.jsonl`
- `reviews.jsonl`
- `manifest.json`

## 0) Install + clone the project
Uses your GitHub repo so the same filters/preprocess code run here and locally.

In [ ]:
!pip -q install datasets pyarrow pandas tqdm

# Clone your repo (public)
!rm -rf SmartPhone-Recommendation-System
!git clone --depth 1 https://github.com/KavishaLP/SmartPhone-Recommendation-System.git
%cd SmartPhone-Recommendation-System
!pip -q install -r requirements.txt

## 1) Configure corpus size
Start small to verify, then increase. `MAX_REVIEW_SCAN = 0` walks the review JSONL until each selected phone has reviews (or the file ends). A positive number is a hard cap.

In [ ]:
MAX_PHONES = 0           # 0 = every real smartphone (not cases); set 40 for a quick test
MAX_REVIEWS = 60         # exact reviews per phone (cap)
MIN_RATINGS = 0          # 0 = no popularity cutoff
MAX_REVIEW_SCAN = 0      # 0 = scan until phones are filled or the JSONL ends
BRANDS = []              # [] = any brand
OUT = "hf_corpus"

print(dict(MAX_PHONES=MAX_PHONES, MAX_REVIEWS=MAX_REVIEWS, MIN_RATINGS=MIN_RATINGS,
           MAX_REVIEW_SCAN=MAX_REVIEW_SCAN, BRANDS=BRANDS, OUT=OUT))

## 2) Build corpus (meta → exact reviews → Step 1 preprocess)

This streams:
- meta parquet: `raw_meta_Cell_Phones_and_Accessories`
- reviews jsonl: `raw/review_categories/Cell_Phones_and_Accessories.jsonl`

Accessories are filtered out. Review `body` is the **exact** dataset text; `cleaned_body` is methodology Step 1.

In [ ]:
import sys
from pathlib import Path

brand_args = []
for b in BRANDS:
    brand_args += ["--brand", b]

cmd = [
    sys.executable, "scripts/build_hf_corpus.py",
    "--max-phones", str(MAX_PHONES),
    "--max-reviews", str(MAX_REVIEWS),
    "--min-ratings", str(MIN_RATINGS),
    "--max-review-scan", str(MAX_REVIEW_SCAN),
    "--out", OUT,
] + brand_args

print("Running:", " ".join(cmd))
!{sys.executable} scripts/build_hf_corpus.py --max-phones {MAX_PHONES} --max-reviews {MAX_REVIEWS} --min-ratings {MIN_RATINGS} --max-review-scan {MAX_REVIEW_SCAN} --out {OUT} {' '.join(brand_args)}

## 3) Inspect + download

In [ ]:
import json
from pathlib import Path
import pandas as pd

out = Path(OUT)
manifest = json.loads((out / "manifest.json").read_text())
print(json.dumps(manifest["counts"], indent=2))
print("\nPhones:")
for p in manifest["phones"]:
    print(f"  {p['asin']:12}  {p['usable_reviews']:4} usable  {p['name']}")

reviews = [json.loads(l) for l in (out / "reviews.jsonl").read_text().splitlines() if l]
df = pd.DataFrame(reviews)
print("\nSample exact review bodies:")
display(df.loc[df["usable"], ["parent_asin", "rating", "body"]].head(5))

In [ ]:
# Zip phones.jsonl + reviews.jsonl + manifest.json and download to your PC
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

out = Path(OUT)
zip_path = Path("hf_corpus.zip")
needed = ["phones.jsonl", "reviews.jsonl", "manifest.json"]
missing = [n for n in needed if not (out / n).exists()]
if missing:
    raise FileNotFoundError(f"Corpus incomplete, missing: {missing}")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as zf:
    for name in needed:
        zf.write(out / name, arcname=f"hf_corpus/{name}")
print(f"Wrote {zip_path.resolve()} ({zip_path.stat().st_size:,} bytes)")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not in Colab — copy hf_corpus.zip (or the folder) onto the research PC.")

## 4) On your PC (after download)

Stop `python run.py serve` if it is running (it locks SQLite). Unzip `hf_corpus.zip` into the project so you have `hf_corpus/phones.jsonl`. Then:

```powershell
cd "d:\4TH YEAR\RESEARCH SETUP\smartphone recommendation"
python run.py import-corpus .\hf_corpus --replace --analyze
python run.py serve
```

Open http://127.0.0.1:8000/ui/ — **Recommend** (weights) and **Phones** (scores). Keep “Minimum analysed reviews” at 0 until the corpus is large.

`--replace` wipes old/demo rows. You do not need `reset-corpus` as well.

### Methodology mapping
| Step | Where it runs |
|---|---|
| 1 Preprocess (dedupe, spam/empty, HTML, normalise, language) | This Colab / `build_hf_corpus.py` |
| 2 Sentence segmentation | `import-corpus --analyze` |
| 3 Aspect extraction | `import-corpus --analyze` (lexicon now; set `LLM_API_KEY` for LLM) |
| 4 Sentiment classification | same |
| 5 Structured aspect-sentiment rows | SQLite `aspect_sentiments` |
| 6 Aspect score aggregation | SQLite `aspect_scores` → Recommend UI |